In [1]:
from nflows.transforms.coupling import ConditionalAffineCouplingTransform


ImportError: cannot import name 'ConditionalAffineCouplingTransform' from 'nflows.transforms.coupling' (/allen/aind/scratch/shuonan.chen/conda_envs/torch_on_gpu_py310/lib/python3.10/site-packages/nflows/transforms/coupling.py)

In [4]:
import numpy as np
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
import scanpy as sc
import matplotlib.pyplot as plt 


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# nflows imports
from nflows.flows import Flow
from nflows.distributions import StandardNormal
from nflows.transforms import CompositeTransform, RandomPermutation, ConditionalAffineCouplingTransform


ImportError: cannot import name 'ConditionalAffineCouplingTransform' from 'nflows.transforms' (/allen/aind/scratch/shuonan.chen/conda_envs/torch_on_gpu_py310/lib/python3.10/site-packages/nflows/transforms/__init__.py)

In [2]:
import trimesh
mesh_LC = trimesh.load_mesh("/allen/aind/scratch/shuonan.chen/scripts/Pons_MERFISH/mesh/LC_ccf_v1_250102 2.obj")
mesh_CD = trimesh.load_mesh("/allen/aind/scratch/shuonan.chen/scripts/Pons_MERFISH/mesh/subCD_ccf_v1_250102 2.obj")
mesh_CV = trimesh.load_mesh("/allen/aind/scratch/shuonan.chen/scripts/Pons_MERFISH/mesh/subCV_ccf_v1_250102 2.obj")
allmeshes = [mesh_LC,mesh_CD,mesh_CV]

In [3]:
filename = '/allen/aind/scratch/shuonan.chen/code/pons_merfish_pipeline/processing/data/adata_mer_subset_2_6k.h5ad'
adata_mer = sc.read_h5ad(filename)
def flip(a, xm):
        return(2*xm-a)

def get_hemi(S_mer, mesh):
    '''
    assume the axis of interest are both on the last axis. 
    '''
    xm = np.min(mesh.vertices[:,-1]) + np.ptp(mesh.vertices[:,-1])/2 # this is the center line to indicate the hemisphere 
    new_coords = S_mer.copy()
    new_coords[:,-1] = np.where(new_coords[:,-1] > xm, flip(new_coords[:,-1],xm), new_coords[:,-1])    
    return(new_coords)



In [6]:

X_mer = adata_mer.X        # Gene expression matrix (2800 x 314)
S_mer = adata_mer.obsm['spatial']  # Spatial coordinates matrix, e.g., (2800 x 2)

S_mer = get_hemi(S_mer, mesh_LC)    


scaler_X = StandardScaler().fit(X_mer)
X_mer_scaled = scaler_X.transform(X_mer)

scaler_S = StandardScaler().fit(S_mer)
S_mer_scaled = S_mer.copy()

In [9]:
foo1 = S_mer_scaled.copy()
foo2 = X_mer_scaled.copy()

In [10]:

# ------------------------------
# Helper: Build a conditioner network.
# This network takes the conditioning input (foo1) and outputs parameters
# (scale and shift) for the coupling transform.
def create_conditioner(context_dim, output_dim, hidden_dim=128):
    # A simple MLP with two hidden layers.
    return nn.Sequential(
        nn.Linear(context_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim)
    )

# ------------------------------
# Build the conditional flow.
# We assume the data (foo2) is 3D and the conditioning (foo1) is also 3D.
def build_conditional_flow(data_dim=3, context_dim=3, num_layers=4):
    transforms = []
    # For a coupling layer, we alternate masks.
    # For 3-dimensional data, one possible pair of masks is:
    masks = [torch.tensor([True, False, True]), torch.tensor([False, True, False])]
    
    for i in range(num_layers):
        # Add a random permutation to mix the coordinates.
        transforms.append(RandomPermutation(features=data_dim))
        
        # Select mask in alternating fashion.
        mask = masks[i % 2]
        # Number of features to be transformed is the count of False in mask.
        n_transformed = (torch.logical_not(mask)).sum().item()
        # The conditioner should output 2 parameters (scale and shift) for each transformed feature.
        conditioner = create_conditioner(context_dim=context_dim, output_dim=2 * n_transformed)
        
        transforms.append(ConditionalAffineCouplingTransform(mask=mask, conditioner=conditioner))
    
    transform = CompositeTransform(transforms)
    base_distribution = StandardNormal(shape=[data_dim])
    flow = Flow(transform, base_distribution)
    return flow

# ------------------------------
# Main training routine.
# We assume you already have foo1 and foo2 as numpy arrays of shape (2651, 3).
def train_conditional_flow(foo1, foo2, num_epochs=1000, batch_size=128, lr=1e-3):
    # Convert numpy arrays to torch tensors.
    foo1_tensor = torch.tensor(foo1, dtype=torch.float32)
    foo2_tensor = torch.tensor(foo2, dtype=torch.float32)
    
    dataset = TensorDataset(foo1_tensor, foo2_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Build our conditional flow model.
    flow = build_conditional_flow(data_dim=3, context_dim=3, num_layers=4)
    optimizer = optim.Adam(flow.parameters(), lr=lr)
    
    for epoch in range(num_epochs):
        total_loss = 0.0
        for cond, target in dataloader:
            optimizer.zero_grad()
            # Compute the negative log-likelihood loss.
            loss = -flow.log_prob(inputs=target, context=cond).mean()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * cond.size(0)
            
        if epoch % 100 == 0:
            avg_loss = total_loss / len(dataset)
            print(f"Epoch {epoch:4d}: Avg Loss = {avg_loss:.4f}")
    
    return flow

# ------------------------------
# Converting foo1 to foo2 using the trained flow.
def convert_with_flow(flow, foo1):
    """
    Given a trained conditional flow and foo1 (as a numpy array), 
    generate corresponding foo2 samples.
    """
    foo1_tensor = torch.tensor(foo1, dtype=torch.float32)
    # Sample from the base distribution and transform conditionally.
    with torch.no_grad():
        foo2_generated = flow.sample(num_samples=foo1_tensor.shape[0], context=foo1_tensor)
    return foo2_generated.numpy()

# ------------------------------
# Example usage:
if __name__ == '__main__':
    import numpy as np
    # For demonstration, we create synthetic data.
    np.random.seed(42)
    # foo1: samples from a standard normal in 3D.
    foo1 = np.random.randn(2651, 3)
    # foo2: a transformed version of foo1 (for example, scaled and shifted).
    foo2 = np.random.randn(2651, 3) * 2.0 + 1.0  # This is just for demonstration.
    
    # Train the conditional flow to learn p(foo2 | foo1)
    flow_model = train_conditional_flow(foo1, foo2, num_epochs=1000, batch_size=128, lr=1e-3)
    
    # Convert foo1 to foo2 using the trained flow.
    foo2_generated = convert_with_flow(flow_model, foo1)
    
    # Print out some statistics for comparison.
    print("Generated foo2 mean:", np.mean(foo2_generated, axis=0))
    print("Target foo2 mean:   ", np.mean(foo2, axis=0))
    print("\nGenerated foo2 covariance:\n", np.cov(foo2_generated, rowvar=False))
    print("Target foo2 covariance:\n", np.cov(foo2, rowvar=False))


ModuleNotFoundError: No module named 'nflows'